### 一、定义天气查询子图 - 智能体

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model

from app.core.config import settings

# 准备模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={"thinking":{"type":"disabled"}}
)

# 准备工具
@tool
def search_weather(city:str):
    """
    根据天气查询实时天气
    :param city:城市
    :return:天气信息
    """
    weather_dict = {
        "北京":"天气晴朗",
        "上海":"多云",
        "东京":"火山爆发"
    }
    return weather_dict.get(city, "无法查询对应城市的天气")

# 创建智能体
weather_agent = create_agent(
    model=model,
    tools=[search_weather],
)


"""
# 调用智能体, 要证是否可用
res = weather_agent.invoke({
    "messages":[HumanMessage(content="北京今天天气怎么样?")]
})
print(res["messages"][-1].text)
"""

### 二、定义购物流程图

In [ ]:
from langgraph.constants import START, END
from langchain_core.messages import AIMessage
from langgraph.graph import MessagesState, StateGraph


# 定义状态
class ShoppingState(MessagesState):
    foo:str

# 定义节点
def shopping_node(state:ShoppingState):
    print(f"获取父图中的intent:{state.get("intent", "无法查询父图中特有的数据~")}")
    print(f"获取父图中的messages:{state.get("messages")}")
    return {
        "foo":"子图中特有的数据",
        "messages":[AIMessage(content="买到了 Apple 18 Duo~")]
    }
# 构件图
shopping_builder = StateGraph(ShoppingState)
shopping_builder.add_node("shopping", shopping_node)

shopping_builder.add_edge(START, "shopping")
shopping_builder.add_edge("shopping", END)

shopping_graph = shopping_builder.compile()

# 调用图
res = shopping_graph.invoke({})
print(f"最终结果: {res}")

### 三、定义父图 - 将前面两个子图作为其中的一个节点使用

In [ ]:
from langgraph.constants import START, END
from typing import Literal
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import MessagesState, StateGraph

from app.core.config import settings
from langchain.chat_models import init_chat_model

# 定义模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

# 1.定义状态
class IntentState(MessagesState):
    intent:str

# 定义节点
# 定义意图识别节点
def intent_node(state:IntentState):
    response:AIMessage = model.invoke(f"""
        根据用户的问题，进行意图识别。
        可选的结果：chat、weather、translation、shopping
        用户问题:{state["messages"][-1].content}
        """)
    return {"intent": response.text}

# 定义闲聊节点
def chat_node(state:IntentState):
    response:AIMessage = model.invoke(state["messages"])
    return {"messages":[response]}
# 定义翻译节点
def translation_node(state: IntentState):
    response:AIMessage = model.invoke(f"""
    将用户的问题进行翻译，如果是中文翻译成英文，如果是英文翻译成中文。
    用户问题:{state["messages"][-1].content}
    """)
    return {
        "messages":[response]
    }

# 定义路由函数
# 3.定义路由函数
def router(state:IntentState) -> Literal["chat", "weather", "translation", "shopping"]:
    return state["intent"]

# 4.定义边，构建智能体
builder = StateGraph(IntentState)

builder.add_node("intent", intent_node)
builder.add_node("chat", chat_node)
builder.add_node("weather", weather_agent)
builder.add_node("translation", translation_node)
builder.add_node("shopping", translation_node)

builder.add_edge(START, "intent")
builder.add_conditional_edges("intent", router)
builder.add_edge("chat", END)
builder.add_edge("weather", END)
builder.add_edge("translation", END)
builder.add_edge("shopping", END)

agent = builder.compile()


# 5.调用智能体
result = agent.invoke({
    "messages":[
        HumanMessage(content="今天北京的天气怎么样？")
        # HumanMessage(content="翻译:我爱睡觉~")
        # HumanMessage(content="你好吗？")
    ]
})

print(f"路由识别结果:{result["intent"]}")
for message in result["messages"]:
    message.pretty_print()